In [109]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.ensemble import RandomForestRegressor
from boruta import BorutaPy
import shap
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

import warnings
warnings.filterwarnings("ignore")

In [110]:
# Configuration
RANDOM_STATE = 42
TARGET_COL = "Degradation (%)"
KNN_NEIGHBORS = 5

In [111]:
df = pd.read_csv("../data/Data.csv")
print(f"Loaded shape: {df.shape}")

df.sample(10, random_state=RANDOM_STATE)

Loaded shape: (162, 3)


,Dye,Rxn Time (min),Removal %
158,CS-3B,10,0.363633
109,BF,5,0.409810
131,Fl,25,0.983809
55,PonS,5,0.298214
94,Reso,20,0.831068
29,R6G,25,0.758447
101,DMMB,25,0.709923
51,Ab,15,0.600005
100,DMMB,20,0.660136
144,MG,0,0.000000


In [112]:
buffer_info = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "n_missing": df.isna().sum(),
})

buffer_info

,dtype,n_unique,n_missing
Dye,object,27,0
Rxn Time (min),int64,6,0
Removal %,float64,136,0


In [113]:
# Duplicate analysis
n_exact_duplicates = df.duplicated().sum()
print(f"Exact duplicate rows: {n_exact_duplicates}")

if n_exact_duplicates > 0:
    duplicates_df = df[df.duplicated(keep=False)].sort_values(by=df.columns.tolist())

Exact duplicate rows: 0


In [114]:
dye_name_map = {
    "MB": "Methylene Blue",
    "MO": "Methyl Orange",
    "Tart": "Tartrazine",
    "RhB": "Rhodamine B",
    "R6G": "Rhodamine 6G",
    "Am": "Amaranth",
    "CV": "Crystal Violet",
    "MaG": "Malachite Green",
    "Ab": "Amido Black 10B",
    "PonS": "Ponceau S",
    "BBR": "Coomassie Brilliant Blue R-250",
    "BBG": "Coomassie Brilliant Blue G-250",
    "CB": "Cibacron Blue 3GA",
    "BPB": "Bromophenol Blue",
    "Erio": "Erioglaucine",
    "Reso": "Resorufin",
    "DR80": "Direct Red 80",
    "Congo": "Congo Red",
    "DMMB": "Dimethylmethylene Blue",
    "BF": "Basic Fuchsin",
    "Fl": "Fluorescein",
    "NBA": "Nile Blue A",
    "OG": "Orange G",
    "CS-3B": "Crocein Scarlet 3B",
    "FBMBSN": "Solvent Blue 38",
    "MG": "Methyl green chloride",
    "BCB": "Brilliant Cresyl Blue ALD",
}

In [115]:
# Validate dye names

if "Dye" not in df.columns:
    raise KeyError(
        f"'Dye' column was not found. Available columns: {df.columns.tolist()}"
    )

unique_dyes = (
    df["Dye"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

missing_from_map = [
    dye for dye in unique_dyes
    if dye not in dye_name_map
]

if missing_from_map:
    print("\nWARNING: These dye codes are missing from dye_name_map:")

    for dye in missing_from_map:
        print(f"  - {dye}")

In [116]:
# Fetch SMILES from PubChem

def fetch_smiles_from_pubchem(dye_code):
    """
    Convert a dye abbreviation to its full name and
    fetch Canonical SMILES from PubChem.
    """

    dye_code = str(dye_code).strip()

    compound_name = dye_name_map.get(dye_code)

    if compound_name is None:
        print(
            f"[NAME MAP FAILED] "
            f"No full name defined for dye code '{dye_code}'"
        )
        return None

    try:
        compounds = pcp.get_compounds(
            compound_name,
            namespace="name"
        )

        if not compounds:
            print(
                f"[PUBCHEM FAILED] "
                f"{dye_code} -> {compound_name}"
            )
            return None

        compound = compounds[0]

        smiles = compound.canonical_smiles

        if not smiles:
            print(
                f"[NO SMILES] "
                f"{dye_code} -> {compound_name}"
            )
            return None

        return smiles

    except Exception as e:

        print(
            f"[ERROR] {dye_code} -> {compound_name}: {e}"
        )

        return None

In [117]:
# Molecular descriptor extraction

def extract_chemical_features_direct(dye_list):
    """
    Fetch molecular structures from PubChem and calculate
    physicochemical descriptors using RDKit.
    """

    features_list = []

    for dye_code in dye_list:

        dye_code = str(dye_code).strip()

        smiles = fetch_smiles_from_pubchem(dye_code)

        if smiles is None:
            continue

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            print(
                f"[RDKIT FAILED] Could not parse SMILES "
                f"for '{dye_code}'"
            )
            continue

        descriptor_row = {
            "Dye": dye_code,
            "Full_Name": dye_name_map[dye_code],
            "SMILES": smiles,

            "MW": round(
                Descriptors.MolWt(mol),
                3
            ),

            "TPSA": round(
                Descriptors.TPSA(mol),
                3
            ),

            "LogP": round(
                Descriptors.MolLogP(mol),
                3
            ),

            "Aromatic_Rings":
                Descriptors.NumAromaticRings(mol),

            "HBD":
                Descriptors.NumHDonors(mol),

            "HBA":
                Descriptors.NumHAcceptors(mol),

            "Rotatable_Bonds":
                Descriptors.NumRotatableBonds(mol),

            "Fraction_Csp3": round(
                Descriptors.FractionCSP3(mol),
                3
            ),

            "Labute_ASA": round(
                rdMolDescriptors.CalcLabuteASA(mol),
                3
            ),

            "Heteroatoms":
                Descriptors.NumHeteroatoms(mol),

            "Formal_Charge":
                Chem.GetFormalCharge(mol),
        }

        features_list.append(descriptor_row)

    return pd.DataFrame(features_list)

In [118]:
# Generate descriptor table

dye_features = extract_chemical_features_direct(
    unique_dyes
)

print("\nDescriptor table shape:")
print(dye_features.shape)

display(dye_features.head())


Descriptor table shape:
(27, 14)


,Dye,Full_Name,SMILES,MW,TPSA,LogP,Aromatic_Rings,HBD,HBA,Rotatable_Bonds,Fraction_Csp3,Labute_ASA,Heteroatoms,Formal_Charge
0,MB,Methylene Blue,CN(C)C1=CC2=C(C=C1)N=C3C=CC(=[N+](C)C)C=C3S2.[...,319.861,19.14,-0.497,1,0,3,1,0.250,134.498,5,0
1,MO,Methyl Orange,CN(C)C1=CC=C(C=C1)N=NC2=CC=C(C=C2)S(=O)(=O)[O-...,327.341,85.16,0.076,2,0,6,4,0.143,151.467,8,0
2,Tart,Tartrazine,C1=CC(=CC=C1N=NC2C(=NN(C2=O)C3=CC=C(C=C3)S(=O)...,534.371,211.92,-9.888,2,0,12,6,0.062,259.629,18,0
3,RhB,Rhodamine B,CCN(CC)C1=CC2=C(C=C1)C(=C3C=CC(=[N+](CC)CC)C=C...,479.020,56.69,2.565,2,1,3,7,0.286,206.270,6,0
4,R6G,Rhodamine 6G,CCNC1=CC2=C(C=C1C)C(=C3C=C(C(=[NH+]CC)C=C3O2)C...,479.020,65.44,1.435,2,2,4,6,0.286,206.169,6,0


In [119]:
# Check unsuccessful PubChem/RDKit lookups

resolved_dyes = set(dye_features["Dye"])

missing_lookup = [
    dye for dye in unique_dyes
    if dye not in resolved_dyes
]

if missing_lookup:

    print("\nDescriptors were NOT generated for:")

    for dye in missing_lookup:
        print(
            f"  {dye:8s} -> "
            f"{dye_name_map.get(dye, 'UNKNOWN')}"
        )

else:
    print("\nAll dyes successfully resolved")


All dyes successfully resolved


In [120]:
df = df.merge(
    dye_features,
    on="Dye",
    how="left",
    validate="many_to_one"
)

In [ ]:
df.head()


,Dye,Rxn Time (min),Removal %,Full_Name,SMILES,MW,TPSA,LogP,Aromatic_Rings,HBD,HBA,Rotatable_Bonds,Fraction_Csp3,Labute_ASA,Heteroatoms,Formal_Charge
0,MB,0,0.000000,Methylene Blue,CN(C)C1=CC2=C(C=C1)N=C3C=CC(=[N+](C)C)C=C3S2.[...,319.861,19.14,-0.497,1,0,3,1,0.25,134.498,5,0
1,MB,5,0.630870,Methylene Blue,CN(C)C1=CC2=C(C=C1)N=C3C=CC(=[N+](C)C)C=C3S2.[...,319.861,19.14,-0.497,1,0,3,1,0.25,134.498,5,0
2,MB,10,0.665654,Methylene Blue,CN(C)C1=CC2=C(C=C1)N=C3C=CC(=[N+](C)C)C=C3S2.[...,319.861,19.14,-0.497,1,0,3,1,0.25,134.498,5,0
3,MB,15,0.707449,Methylene Blue,CN(C)C1=CC2=C(C=C1)N=C3C=CC(=[N+](C)C)C=C3S2.[...,319.861,19.14,-0.497,1,0,3,1,0.25,134.498,5,0
4,MB,20,0.742452,Methylene Blue,CN(C)C1=CC2=C(C=C1)N=C3C=CC(=[N+](C)C)C=C3S2.[...,319.861,19.14,-0.497,1,0,3,1,0.25,134.498,5,0


In [125]:
X = df.drop(["Dye", "Removal %", "Full_Name", "SMILES"], axis=1)
y = df["Removal %"]

In [126]:
X.head()

,Rxn Time (min),MW,TPSA,LogP,Aromatic_Rings,HBD,HBA,Rotatable_Bonds,Fraction_Csp3,Labute_ASA,Heteroatoms,Formal_Charge
0,0,319.861,19.14,-0.497,1,0,3,1,0.25,134.498,5,0
1,5,319.861,19.14,-0.497,1,0,3,1,0.25,134.498,5,0
2,10,319.861,19.14,-0.497,1,0,3,1,0.25,134.498,5,0
3,15,319.861,19.14,-0.497,1,0,3,1,0.25,134.498,5,0
4,20,319.861,19.14,-0.497,1,0,3,1,0.25,134.498,5,0
